# Linear optimization with PuLP
[PuLP](https://coin-or.github.io/pulp/) expresses allocation decisions as variables, an objective and constraints. This notebook uses CBC to solve three small examples:

1. Ship beer from two warehouses to five bars at minimum cost.
2. Check whether item quantities fit into three storage boxes.
3. Find the minimum box count for six indivisible items and compare it with a capacity estimate.


## 1. The Beer Distribution Problem

A brewery has two warehouses and five bars to supply. Each warehouse has a limited amount of beer in stock, each bar has a minimum amount it needs delivered, and every warehouse-to-bar route has its own shipping cost. The question: **which routes should carry how much beer so that every bar's demand is met without exceeding any warehouse's supply, at the lowest total shipping cost?**

This is the textbook *transportation problem*. With warehouses $W$, bars $B$, supply $s_w$, demand $d_b$, and per-unit shipping cost $c_{wb}$, it's written as:

$$\min \sum_{w \in W} \sum_{b \in B} c_{wb} \, x_{wb}$$

subject to

$$\sum_{b \in B} x_{wb} \le s_w \quad \forall w \in W \qquad \text{(can't ship more than a warehouse holds)}$$

$$\sum_{w \in W} x_{wb} \ge d_b \quad \forall b \in B \qquad \text{(every bar's order must be filled)}$$

$$x_{wb} \ge 0 \text{ and integer} \qquad \text{(no fractional crates)}$$

### Setting up the data


In [1]:
import pandas as pd
from pulp import LpProblem, LpMinimize, LpVariable, LpInteger, LpStatus, lpSum, makeDict, value

# Supply nodes (warehouses) and how much beer each one holds
warehouses = ["A", "B"]
supply = {"A": 4000, "B": 5000}

# Demand nodes (bars) and how much beer each one needs
bars = ["1", "2", "3", "4", "5"]
demand = {"1": 1900, "2": 900, "3": 1800, "4": 2200, "5": 1700}

# Cost (per unit) of shipping from each warehouse to each bar
cost_table = [
    [2, 4, 5, 2, 1],  # from warehouse A
    [3, 1, 3, 2, 3],  # from warehouse B
]
costs = makeDict([warehouses, bars], cost_table, 0)

pd.DataFrame(cost_table, index=warehouses, columns=bars).rename_axis("Warehouse").rename_axis("Bar", axis="columns")


Bar,1,2,3,4,5
Warehouse,,,,,
A,2,4,5,2,1
B,3,1,3,2,3


### Building the model

PuLP's job is to turn the math above into a concrete object: one decision variable per warehouse-bar route, one objective function, and one constraint per supply/demand limit.


In [2]:
# One decision variable per (warehouse, bar) route: how many units to ship
routes = [(w, b) for w in warehouses for b in bars]
ship = LpVariable.dicts("Route", (warehouses, bars), lowBound=0, cat=LpInteger)

prob = LpProblem("Beer_Distribution_Problem", LpMinimize)

# Objective: minimize total shipping cost across every route
prob += lpSum(ship[w][b] * costs[w][b] for w, b in routes), "Total_Transportation_Cost"

# Each warehouse can't ship more than it has in stock
for w in warehouses:
    prob += lpSum(ship[w][b] for b in bars) <= supply[w], f"Supply_Limit_{w}"

# Each bar must receive at least what it ordered
for b in bars:
    prob += lpSum(ship[w][b] for w in warehouses) >= demand[b], f"Demand_Requirement_{b}"


### Inspect the formulation
Printing the model shows the objective, constraints and variable bounds before solving.


In [3]:
print(prob)


Beer_Distribution_Problem:
MINIMIZE
2*Route_A_1 + 4*Route_A_2 + 5*Route_A_3 + 2*Route_A_4 + 1*Route_A_5 + 3*Route_B_1 + 1*Route_B_2 + 3*Route_B_3 + 2*Route_B_4 + 3*Route_B_5 + 0.0
SUBJECT TO
Supply_Limit_A: Route_A_1 + Route_A_2 + Route_A_3 + Route_A_4 + Route_A_5
 <= 4000

Supply_Limit_B: Route_B_1 + Route_B_2 + Route_B_3 + Route_B_4 + Route_B_5
 <= 5000

Demand_Requirement_1: Route_A_1 + Route_B_1 >= 1900

Demand_Requirement_2: Route_A_2 + Route_B_2 >= 900

Demand_Requirement_3: Route_A_3 + Route_B_3 >= 1800

Demand_Requirement_4: Route_A_4 + Route_B_4 >= 2200

Demand_Requirement_5: Route_A_5 + Route_B_5 >= 1700

VARIABLES
0 <= Route_A_1 Integer
0 <= Route_A_2 Integer
0 <= Route_A_3 Integer
0 <= Route_A_4 Integer
0 <= Route_A_5 Integer
0 <= Route_B_1 Integer
0 <= Route_B_2 Integer
0 <= Route_B_3 Integer
0 <= Route_B_4 Integer
0 <= Route_B_5 Integer



### Solve and read the result
CBC returns an Optimal status and a transportation cost of 16,200 for the stated supplies, demands and route costs.


In [5]:
from pulp import PULP_CBC_CMD

prob.solve(PULP_CBC_CMD(msg=False))

print("Status:", LpStatus[prob.status])
print("Total cost of transportation:", value(prob.objective))


Status: Optimal
Total cost of transportation: 16200.0


In [6]:
# Show quantities by route; darker cells carry more units.
plan = pd.DataFrame(
    {b: [ship[w][b].varValue for w in warehouses] for b in bars},
    index=warehouses,
).rename_axis("Warehouse").rename_axis("Bar", axis="columns")

plan.style.background_gradient(cmap="Blues").format("{:.0f}")


Bar,1,2,3,4,5
Warehouse,,,,,
A,1900,0,0,400,1700
B,0,900,1800,1800,0


Warehouse A uses all 4,000 units, supplying bars 1, 4 and 5. Warehouse B supplies 4,500 units to bars 2, 3 and 4, leaving 500 unused. Every bar's demand is met at a total cost of 16,200.

## 2. Allocate items to storage boxes
This example checks whether item quantities fit into fixed-capacity boxes. Units can be split across boxes. Every allocation has the same per-unit cost, so the objective finds a feasible arrangement rather than favoring particular routes.


In [7]:
# Items to place, and how many units of each exist
item_ids = ["1", "2", "3", "4", "5", "6"]
item_quantity = {"1": 10, "2": 20, "3": 20, "4": 20, "5": 10, "6": 10}

# Boxes available, and the capacity of each
boxes = ["Box1", "Box2", "Box3"]
box_capacity = {"Box1": 40, "Box2": 40, "Box3": 40}

print(f"Total units to place: {sum(item_quantity.values())}")
print(f"Total box capacity:   {sum(box_capacity.values())}")


Total units to place: 90
Total box capacity:   120


The items total 90 units and the boxes hold 120. Since units can be split and there are no item/box restrictions, capacity is sufficient. The solver checks the individual constraints and returns an allocation.


In [8]:
allocation_routes = [(box, item) for box in boxes for item in item_ids]
place = LpVariable.dicts("Place", (boxes, item_ids), lowBound=0, cat=LpInteger)

alloc_prob = LpProblem("Storage_Allocation", LpMinimize)

# Equal per-unit costs make this a feasibility check.
alloc_prob += lpSum(place[box][item] for box, item in allocation_routes), "Units_Placed"

# A box can never hold more than its capacity
for box in boxes:
    alloc_prob += lpSum(place[box][item] for item in item_ids) <= box_capacity[box], f"Capacity_{box}"

# Every unit of every item must end up in some box
for item in item_ids:
    alloc_prob += lpSum(place[box][item] for box in boxes) >= item_quantity[item], f"Fulfil_Item_{item}"

alloc_prob.solve(PULP_CBC_CMD(msg=False))
print("Status:", LpStatus[alloc_prob.status])


Status: Optimal


In [10]:
result = pd.DataFrame(
    {item: [place[box][item].varValue for box in boxes] for item in item_ids},
    index=boxes,
).rename_axis("Box").rename_axis("Item", axis="columns")

used = result.sum(axis=1).astype(int)
capacity = pd.Series(box_capacity).astype(int)
result["Used / Capacity"] = used.astype(str) + " / " + capacity.astype(str)

result


Item,1,2,3,4,5,6,Used / Capacity
Box,,,,,,,
Box1,0.0,20.0,0.0,20.0,0.0,0.0,40 / 40
Box2,10.0,0.0,0.0,0.0,0.0,10.0,20 / 40
Box3,0.0,0.0,20.0,0.0,10.0,0.0,30 / 40


The solver finds a feasible allocation of all 90 units, leaving 30 units of capacity unused. Equal route costs mean several arrangements can have the same objective value.

## 3. Minimum box count
This example packs six indivisible items. Dividing total size by box capacity gives a lower bound. It may underestimate the number of boxes because individual items cannot be split to fill every gap.


In [11]:
import math

items = pd.DataFrame(
    [["item1", 33], ["item2", 14], ["item3", 27], ["item4", 23], ["item5", 16], ["item6", 20]],
    columns=["item", "size"],
)
box_size = 40

total_size = items["size"].sum()
boxes_needed_exact = total_size / box_size
boxes_needed_heuristic = math.ceil(boxes_needed_exact)
spare_capacity = boxes_needed_heuristic * box_size - total_size

print(f"Total size of all items: {total_size}")
print(f"Heuristic (round up total / box size): {boxes_needed_heuristic} boxes")
print(f"Implied spare capacity: {spare_capacity} units ({spare_capacity / (boxes_needed_heuristic * box_size):.1%} of total space)")


Total size of all items: 133
Heuristic (round up total / box size): 4 boxes
Implied spare capacity: 27 units (16.9% of total space)


### Integer formulation
A binary variable $x_{ij}$ assigns item $i$ to box $j$, and $y_j$ indicates whether box $j$ is used.

$$\min \sum_j y_j \quad \text{subject to} \quad \sum_j x_{ij}=1\ \forall i, \qquad \sum_i \text{size}_i x_{ij} \le \text{box size}\,y_j\ \forall j$$

One candidate box per item is sufficient for these data because each item fits in a box individually. The objective leaves unnecessary boxes unused.


In [12]:
from pulp import LpBinary

item_ids = items["item"].tolist()
sizes = dict(zip(items["item"], items["size"]))
candidate_boxes = [f"box{i + 1}" for i in range(len(item_ids))]  # one per item is always enough

pack_prob = LpProblem("Bin_Packing", LpMinimize)

in_box = LpVariable.dicts("assign", (item_ids, candidate_boxes), cat=LpBinary)
box_used = LpVariable.dicts("used", candidate_boxes, cat=LpBinary)

# Minimize the number of boxes actually switched "on"
pack_prob += lpSum(box_used[j] for j in candidate_boxes), "Boxes_Used"

# Every item goes into exactly one box
for i in item_ids:
    pack_prob += lpSum(in_box[i][j] for j in candidate_boxes) == 1, f"Place_{i}"

# A box can't exceed its capacity, and can't hold anything unless it's "used"
for j in candidate_boxes:
    pack_prob += lpSum(sizes[i] * in_box[i][j] for i in item_ids) <= box_size * box_used[j], f"Capacity_{j}"

pack_prob.solve(PULP_CBC_CMD(msg=False))

boxes_needed_exact = int(value(pack_prob.objective))
print("Status:", LpStatus[pack_prob.status])
print(f"Exact minimum number of boxes: {boxes_needed_exact}")
print(f"Heuristic estimate was:        {boxes_needed_heuristic}")


Status: Optimal
Exact minimum number of boxes: 4
Heuristic estimate was:        4


In [13]:
for box in candidate_boxes:
    if box_used[box].varValue == 1:
        contents = [i for i in item_ids if in_box[i][box].varValue == 1]
        used = sum(sizes[i] for i in contents)
        print(f"{box}: {contents} -> {used}/{box_size} used, {box_size - used} spare")


box1: ['item3'] -> 27/40 used, 13 spare
box3: ['item4', 'item5'] -> 39/40 used, 1 spare
box4: ['item2', 'item6'] -> 34/40 used, 6 spare
box6: ['item1'] -> 33/40 used, 7 spare


The capacity estimate and integer solver both give four boxes for these item sizes. Matching the lower bound confirms the minimum here. Different item sizes may require more boxes even when total capacity appears sufficient; an item larger than a box would make the model infeasible.

The three examples differ in what is being optimized: route cost, feasibility with fixed boxes, and the number of boxes used. The next step for a real allocation problem would be to add its item compatibility and handling constraints before interpreting the plan.
